<a href="https://colab.research.google.com/github/Asgorreaj/ai-model-evaluation-machine-learning-notebook-llama3/blob/main/machine-learning-ai-notebook-llama3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><img src="https://images2.mobilissimo.ro/ejM/662234d3d923f.jpg" width=400></center>

# Introduction

Llama 3 is the latest model from Meta. Let's try and see what we can do with it.  

Will test now the following model:
* **Model**: Llama3
* **Version**: 8b-chat-hf
* **Framework**: Transformers
* **Version**: V1

This is what we will test:

* Simple prompts with general information questions
* Poetry (haiku, sonets) writing
* Code writing (Python, C++, Java)
* Software design (simple problems)
* Multi-parameter questions
* Chain of reasoning
* A more complex reasoning problem

I intend to learn from this experience so that I can then build then someting a bit more complex.

Note: this notebook is organized to facilitate a comparison of **Llama3** with **Gemma**. Corresponding notebook is [Prompt Gemma like a Pro](https://www.kaggle.com/code/gpreda/prompt-gemma-like-a-pro). At the end of current notebook we are reviewing the performance of the two notebooks for the list of tasks submitted to both.

Notebook Courtesy: Gabriel Preda
[Prompt Llama 3like a Pro](https://www.kaggle.com/code/gpreda/prompting-llama-3-like-a-pro)

Model: [Llama 3](https://www.kaggle.com/models/metaresearch/llama-3/Transformers/8b-chat-hf/1)

More [visualizations](https://jalammar.github.io/illustrated-transformer/)

In [ ]:
from IPython.display import Image
from IPython.core.display import HTML
Image(url= "https://miro.medium.com/v2/resize:fit:1400/format:webp/1*abmWOcJmVAzDEwxtls5TXQ.png")


# Preparation

We import the libraries we need, and we set the model to be ready for testing.

## Import packages

In [ ]:
from time import time
import torch
import transformers
from transformers import AutoTokenizer
from IPython.display import display, Markdown

## Define model

This will take some time (loading checkpoint shards, cca. 2 min.).

In [ ]:
model = "/kaggle/input/llama-3/transformers/8b-chat-hf/1"

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

2025-02-15 11:14:36.563362: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-15 11:14:36.563506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-15 11:14:36.914881: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
!ls /kaggle/input/llama-3/transformers/8b-chat-hf/1

LICENSE				  model-00003-of-00004.safetensors
USE_POLICY.md			  model-00004-of-00004.safetensors
__init__.py			  model.py
config.json			  model.safetensors.index.json
eval_details.md			  requirements.txt
example_chat_completion.py	  setup.py
example_text_completion.py	  special_tokens_map.json
generation.py			  test_tokenizer.py
generation_config.json		  tokenizer.json
model-00001-of-00004.safetensors  tokenizer.py
model-00002-of-00004.safetensors  tokenizer_config.json


## Prepare query function

In [ ]:
def query_model(
        system_message,
        user_message,
        temperature=0.7,
        max_length=1024
        ):
    start_time = time()
    user_message = "Question: " + user_message + " Answer:"
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
        ]
    prompt = pipeline.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
        )
    terminators = [
        pipeline.tokenizer.eos_token_id,
        pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]
    sequences = pipeline(
        prompt,
        do_sample=True,
        top_p=0.9,
        temperature=temperature,
        #num_return_sequences=1,
        eos_token_id=terminators,
        max_new_tokens=max_length,
        return_full_text=False,
        pad_token_id=pipeline.model.config.eos_token_id
    )
    #answer = f"{sequences[0]['generated_text'][len(prompt):]}\n"
    answer = sequences[0]['generated_text']
    end_time = time()
    ttime = f"Total time: {round(end_time-start_time, 2)} sec."

    return user_message + " " + answer  + " " +  ttime


system_message = """
You are an AI assistant designed to answer simple questions.
Please restrict your answer to the exact question asked.
"""

## Utility function for output format

In [ ]:
def colorize_text(text):
    for word, color in zip(["Reasoning", "Question", "Answer", "Total time"], ["blue", "red", "green", "magenta"]):
        text = text.replace(f"{word}:", f"\n\n**<font color='{color}'>{word}:</font>**")
    return text

# Test with few simple geography and history questions

In [ ]:
response = query_model(
    system_message,
    user_message="What is the surface temperature of the Moon?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** What is the surface temperature of the Moon? 

**<font color='green'>Answer:</font>** The surface temperature of the Moon varies greatly depending on the time of day and the location on the Moon. During the day, the temperature can reach as high as 253°F (122°C), while at night it can drop to as low as -243°F (-153°C). 

**<font color='magenta'>Total time:</font>** 7.53 sec.

Let's try also with a different prompt.

In [ ]:
response = query_model(
    system_message,
    user_message="What is the surface of France?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** What is the surface of France? 

**<font color='green'>Answer:</font>** The surface area of France is approximately 551,500 square kilometers (213,011 square miles). 

**<font color='magenta'>Total time:</font>** 1.44 sec.

It doesn't look like a well crafted prompt helps too much.  Let's get back to our initial prompt pattern for more questions.

In [ ]:
response = query_model(
    system_message,
    user_message="When was the 30 years war?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** When was the 30 years war? 

**<font color='green'>Answer:</font>** The Thirty Years' War was from 1618 to 1648. 

**<font color='magenta'>Total time:</font>** 1.13 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="What is graphe paranomon?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** What is graphe paranomon? 

**<font color='green'>Answer:</font>** Graphe paranomon is a type of ancient Greek legal procedure, literally meaning "charge of illegality". It was a way for the Athenian assembly to impeach and punish magistrates who had acted illegally or unconstitutionally. 

**<font color='magenta'>Total time:</font>** 3.28 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="Who was the next shogun after Tokugawa Ieyasu?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Who was the next shogun after Tokugawa Ieyasu? 

**<font color='green'>Answer:</font>** Tokugawa Hidetada. 

**<font color='magenta'>Total time:</font>** 0.64 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="What was the name of the Chinese dinasty during 1st century B.C.?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** What was the name of the Chinese dinasty during 1st century B.C.? 

**<font color='green'>Answer:</font>** The Han Dynasty. 

**<font color='magenta'>Total time:</font>** 0.37 sec.

Let's try some elementary questions about American history.

In [ ]:
response = query_model(
    system_message,
    user_message="Who was the first American president?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Who was the first American president? 

**<font color='green'>Answer:</font>** George Washington. 

**<font color='magenta'>Total time:</font>** 0.3 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="When took place the Civil War in United States of America?",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** When took place the Civil War in United States of America? 

**<font color='green'>Answer:</font>** The American Civil War took place from 1861 to 1865. 

**<font color='magenta'>Total time:</font>** 1.11 sec.

# Let's write poetry

In [ ]:
system_message = """
You are an AI assistant designed to write poetry.
Please answer with a haiku format (17 words poems).
Question: {question}
Answer:
"""
response = query_model(
    system_message,
    user_message="Please write a poem about Boris Becker wins in tennis",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a poem about Boris Becker wins in tennis 

**<font color='green'>Answer:</font>** Golden racket swings
Boris Becker's mighty serve
Victory's sweet song 

**<font color='magenta'>Total time:</font>** 1.18 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="Please write a poem about Shakespeare being lame at playing poker",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a poem about Shakespeare being lame at playing poker 

**<font color='green'>Answer:</font>** Bard's bluffing fails hard
Shakespeare's poker face a lie
Lame, his royal flush 

**<font color='magenta'>Total time:</font>** 1.6 sec.

In [ ]:
system_message = """
You are an AI assistant designed to write poetry.
Please answer with a short poem, with rime, in the style of Shakespeare's poems.
Question: {question}
Answer:
"""
response = query_model(
    system_message,
    user_message="Please write a poem about Nadia Comaneci winning Montreal Olympiad",
    temperature=0.1,
    max_length=512)
display(Markdown(colorize_text(response)))

--- Logging error ---
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 1100, in emit
    msg = self.format(record)
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 943, in format
    return fmt.format(record)
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 678, in format
    record.message = record.getMessage()
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 368, in getMessage
    msg = msg % self.args
TypeError: not all arguments converted during string formatting
Call stack:
  File "/opt/conda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/conda/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.10/site-packages/traitlets/config/application.py", line 104



**<font color='red'>Question:</font>** Please write a poem about Nadia Comaneci winning Montreal Olympiad 

**<font color='green'>Answer:</font>** Fair Nadia, with thy nimble might,
Didst conquer Montreal's Olympic light.
Thy bars, a canvas for thy skill so bright,
Didst shine like stars on a summer's night.

Thy score, a perfect ten, didst make us stare,
As if the gods themselves didst declare
Thy triumph, and thy name didst echo there,
Throughout the land, and far and near.

Thy Romanian heart, with passion's fire,
Didst drive thee on, to reach the highest tier.
And when thou landedst, with arms outstretched,
The crowd, in awe, didst thy great deed beset.

Oh, Nadia, thy name shall live on high,
In annals of sport, and in the Olympic sky.
For thou, a shining star, didst light the way,
And in thy triumph, we didst behold a brighter day. 

**<font color='magenta'>Total time:</font>** 12.72 sec.

# Math problem and Python code

In [ ]:
system_message = """
You are an AI assistant designed to write simple Python code.
Please answer with the listing of the Python code.
Question: {question}
Answer:
"""
response = query_model(
    system_message,
    user_message="Please write a function in Python to calculate the area of a circle of radius r",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a function in Python to calculate the area of a circle of radius r 

**<font color='green'>Answer:</font>** Here is the Python code for the function:

```
import math

def circle_area(r):
    return math.pi * (r ** 2)
``` 

**<font color='magenta'>Total time:</font>** 2.23 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="Please write a function in Python to order a list",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a function in Python to order a list 

**<font color='green'>Answer:</font>** Here is a Python function that orders a list:

```
def order_list(lst):
    return sorted(lst)
``` 

**<font color='magenta'>Total time:</font>** 1.67 sec.

# Software design

In [ ]:
response = query_model(
    system_message,
    user_message="""Please write a class in Python
                        to model a phone book (storing name, surname, address, phone)
                        with add, delete, order by name, search operations.
                        The class should store a list of contacts, each
                        with name, surname, address, phone information stored.
                        """,
    temperature=0.1,
    max_length=1024)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a class in Python 
                        to model a phone book (storing name, surname, address, phone) 
                        with add, delete, order by name, search operations.
                        The class should store a list of contacts, each
                        with name, surname, address, phone information stored.
                         

**<font color='green'>Answer:</font>** Here is the Python code for the PhoneBook class:

```
class PhoneBook:
    def __init__(self):
        self.contacts = []

    def add_contact(self, name, surname, address, phone):
        self.contacts.append({"name": name, "surname": surname, "address": address, "phone": phone})

    def delete_contact(self, name):
        for contact in self.contacts:
            if contact["name"] == name:
                self.contacts.remove(contact)
                print(f"Contact {name} deleted successfully.")
                return
        print(f"Contact {name} not found.")

    def order_by_name(self):
        self.contacts.sort(key=lambda x: x["name"])

    def search_contact(self, name):
        for contact in self.contacts:
            if contact["name"] == name:
                return contact
        return None

    def display_contacts(self):
        for contact in self.contacts:
            print(f"Name: {contact['name']}, Surname: {contact['surname']}, Address: {contact['address']}, Phone: {contact['phone']}")


# Example usage:
phone_book = PhoneBook()
phone_book.add_contact("John", "Doe", "123 Main St", "1234567890")
phone_book.add_contact("Jane", "Smith", "456 Elm St", "9876543210")
phone_book.order_by_name()
phone_book.display_contacts()
print(phone_book.search_contact("John"))
phone_book.delete_contact("Jane")
phone_book.display_contacts()
``` 

**<font color='magenta'>Total time:</font>** 21.46 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="""Please write a small Python module that creates a REST API service
                        with two endpoints:
                        * get_status (GET)
                        * prediction (POST)
                        The prediction endpoint receives the payload, extract three fields: city, street and number
                        and returns a field called price_estimate

                        """,
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a small Python module that creates a REST API service
                        with two endpoints: 
                        * get_status (GET)
                        * prediction (POST)
                        The prediction endpoint receives the payload, extract three fields: city, street and number
                        and returns a field called price_estimate
                        
                         

**<font color='green'>Answer:</font>** Here is a simple Python module using Flask that creates a REST API service with the two endpoints you requested:

```
from flask import Flask, request, jsonify
import json

app = Flask(__name__)

# Sample data for prediction
data = {
    "New York": {"street": {"1st": 100, "2nd": 120, "3rd": 150}, 
                 "Los Angeles": {"1st": 80, "2nd": 90, "3rd": 110}, 
                 "Chicago": {"1st": 70, "2nd": 80, "3rd": 100}
}

@app.route('/get_status', methods=['GET'])
def get_status():
    return jsonify({'status': 'OK'})

@app.route('/prediction', methods=['POST'])
def prediction():
    if request.is_json:
        data = request.get_json()
        city = data.get('city')
        street = data.get('street')
        number = data.get('number')
        
        if city and street and number:
            if city in data and street in data[city] and number in data[city][street]:
                return jsonify({'price_estimate': data[city][street][number]})
            else:
                return jsonify 

**<font color='magenta'>Total time:</font>** 17.78 sec.

# C++ code

In [ ]:
system_message = """
You are an AI assistant designed to write simple C++ code.
Please answer with the listing of the C++ code.
Question: {question}
Answer:
"""
response = query_model(
    system_message,
    user_message="Please write a function in C++ to calculate the area of a circle of radius r",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a function in C++ to calculate the area of a circle of radius r 

**<font color='green'>Answer:</font>** Here is the C++ code for the function to calculate the area of a circle:

```cpp
#include<iostream>
using namespace std;

double circleArea(double r) {
    return 3.14 * r * r;
}

int main() {
    double radius;
    cout << "Enter the radius of the circle: ";
    cin >> radius;
    cout << "The area of the circle is: " << circleArea(radius);
    return 0;
}
``` 

**<font color='magenta'>Total time:</font>** 6.68 sec.

In [ ]:
system_message = """
You are an AI assistant designed to write simple C++ code.
Please answer with the listing of the C++ code.
Question: {question}
Answer:
"""
response = query_model(
    system_message,
    user_message="Please write a function in C++ to calculate the volume of a cylinder with radius r and height h",
    temperature=0.1,
    max_length=512)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a function in C++ to calculate the volume of a cylinder with radius r and height h 

**<font color='green'>Answer:</font>** Here is the C++ code for the function to calculate the volume of a cylinder:

```cpp
#include<iostream>
using namespace std;

double cylinderVolume(double r, double h) {
    return 3.14 * r * r * h;
}

int main() {
    double r, h;
    cout << "Enter the radius of the cylinder: ";
    cin >> r;
    cout << "Enter the height of the cylinder: ";
    cin >> h;
    cout << "The volume of the cylinder is: " << cylinderVolume(r, h) << endl;
    return 0;
}
``` 

**<font color='magenta'>Total time:</font>** 8.71 sec.

In [ ]:
response = query_model(
    system_message,
    user_message="Please write a function in C++ to order a list",
    temperature=0.1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** Please write a function in C++ to order a list 

**<font color='green'>Answer:</font>** Here is a simple implementation of a function in C++ that sorts a list of integers in ascending order using the built-in `sort` function from the `<algorithm>` library:

```cpp
#include <iostream>
#include <algorithm>
#include <vector>

void sortList(std::vector<int>& list) {
    std::sort(list.begin(), list.end());
}

int main() {
    std::vector<int> list = {4, 2, 9, 6, 5, 1, 3, 7, 8};
    std::cout << "Original list: ";
    for (int i : list) {
        std::cout << i << " ";
    }
    std::cout << std::endl;

    sortList(list);

    std::cout << "Sorted list: ";
    for (int i : list) {
        std::cout << i << " ";
    }
    std::cout << std::endl;

    return 0;
}
```

This code defines a function `sortList` that takes a reference to a `std::vector<int>` as its argument. It uses the `std::sort` function to sort the list in ascending order. The `main` function demonstrates how to use this function by 

**<font color='magenta'>Total time:</font>** 17.78 sec.

# Multiple parameters questions

## Best food in France

In [ ]:
system_message = """
You are an AI assistant designed to answer questions with parameters.
Return the answer formated nicely, for example with bullet points.
"""
user_message = """
What are the {adjective} {number} {items} from {place}?
"""
response = query_model(
    system_message,
    user_message.format(
    adjective="best",
    number="3",
    items="food",
    place="France"
    ),
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
What are the best 3 food from France?
 

**<font color='green'>Answer:</font>** Bonjour! France, the land of culinary delights! Here are the top 3 French foods that are sure to tantalize your taste buds:

• **Croissants**: Flaky, buttery, and oh-so-French, croissants are a classic breakfast treat that's perfect with a café au lait or a glass of orange juice.

• **Coq au Vin**: A classic dish from the Burgundy region, Coq au Vin is a rich and flavorful braised chicken dish cooked in red wine, mushrooms, onions, and bacon. It's a hearty and comforting meal that's sure to satisfy.

• **Macarons**: These delicate meringue-based cookies are a French specialty that's both beautiful and delicious. With a wide range of flavors and colors, macarons are the perfect treat to indulge in after a long day.

Bon appétit! 

**<font color='magenta'>Total time:</font>** 12.36 sec.

## Best attractions in Italy

In [ ]:
response = query_model(
    system_message,
    user_message.format(
    adjective="best",
    number="five",
    items="attractions",
    place="Italy"
    ),
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
What are the best five attractions from Italy?
 

**<font color='green'>Answer:</font>** Italy! The land of love, art, architecture, and delicious food. Here are the top 5 attractions in Italy that you shouldn't miss:

• **Colosseum** (Rome): This ancient amphitheater is one of Rome's most iconic landmarks and a testament to the city's rich history. Learn about the gladiators who fought here and the emperors who ruled from this very spot.

• **Canal Grande** (Venice): Take a romantic gondola ride or stroll along the Riva degli Schiavoni to admire the stunning beauty of Venice's main canal. Admire the ornate bridges, ornate palaces, and picturesque piazzas.

• **Uffizi Gallery** (Florence): This world-renowned museum is home to some of the most iconic works of art in the world, including Botticelli's "The Birth of Venus" and Leonardo da Vinci's "Annunciation." Be prepared for crowds, but the views are well worth it.

• **Trevi Fountain** (Rome): This beautiful baroque fountain is one of the most famous in the world, and for good reason. Toss a coin into the water to ensure your return to Rome, and enjoy the 

**<font color='magenta'>Total time:</font>** 17.72 sec.

## Most affordable places to stay in Spain

In [ ]:
response = query_model(
    system_message,
    user_message.format(
    adjective="most affordable",
    number="two",
    items="places to retire",
    place="Spain"
    ),
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
What are the most affordable two places to retire from Spain?
 

**<font color='green'>Answer:</font>** Retiring in Spain can be a wonderful idea, with its warm climate, rich culture, and affordable cost of living. Here are the most affordable places to retire in Spain:

• **Granada**: Known for its stunning Alhambra palace and vibrant cultural scene, Granada is a great place to retire. The cost of living is relatively low, with a one-bedroom apartment averaging €600-€800 per month. The city has a warm climate, and the Sierra Nevada mountains offer great hiking and skiing opportunities.

• **Alicante**: Located on the Costa Blanca, Alicante is a popular retirement destination due to its beautiful beaches, mild climate, and low cost of living. A one-bedroom apartment in the city center costs around €400-€600 per month. Alicante also has a rich cultural scene, with many festivals and events throughout the year.

Other factors to consider when retiring in Spain include:

* Healthcare: Spain has a well-regarded public healthcare system, and many private healthcare options are also available.
* Language: While Spanish is the official language, many Spaniards speak English, especially in tourist areas. However, learning some Spanish can greatly enhance your experience.
* Lifestyle: Spain is known for its relaxed pace of life, and many retirees enjoy 

**<font color='magenta'>Total time:</font>** 17.72 sec.

## Less known but great places to stay in Romania

In [ ]:
response = query_model(
    system_message,
    user_message.format(
    adjective="Less known but great",
    number="4",
    items="places to stay",
    place="Romania"
    ),
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
What are the Less known but great 4 places to stay from Romania?
 

**<font color='green'>Answer:</font>** **Less Known but Great Places to Stay in Romania**

Romania is a treasure trove of hidden gems, and when it comes to accommodations, you'll find some amazing options beyond the usual tourist hotspots. Here are four lesser-known but great places to stay in Romania:

• **The Village Museum Guesthouse** (Bucharest): This unique guesthouse is located within the Bucharest Village Museum, a living open-air museum showcasing traditional Romanian architecture. The guesthouse offers cozy rooms and suites, each decorated with antique furniture and artifacts. Enjoy a peaceful stay amidst the rustic charm of rural Romania.

• **Casa cu Tei** (Sighisoara): This charming guesthouse is situated in the heart of Sighisoara's historic center, within walking distance to the famous Clock Tower and Church on the Hill. The name "Casa cu Tei" translates to "House with Limes" due to the beautiful lime trees surrounding the property. The guesthouse offers comfortable rooms and a cozy atmosphere.

• **Pensiunea La Ceaun** (Targu Mures): This family-run pension is located in the picturesque town of Targu Mures, known for its beautiful Baroque architecture. The guesthouse features cozy rooms, a lovely 

**<font color='magenta'>Total time:</font>** 17.79 sec.

## Best commedies by Shakespeare

In [ ]:
response = query_model(
    system_message,
    user_message.format(
    adjective="best",
    number="3",
    items="commedies",
    place="Shakespeare entire creation"
    ),
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
What are the best 3 commedies from Shakespeare entire creation?
 

**<font color='green'>Answer:</font>** Here are three of the most popular and enduring comedies from Shakespeare's works:

• **A Midsummer Night's Dream**: A magical and whimsical play that explores the world of fairies, love, and mischief in the forest. It's a delightful and enchanting tale that has captivated audiences for centuries.

• **Twelfth Night**: A witty and charming play that follows the story of a shipwrecked heroine, Viola, and her twin brother, Sebastian. It's a clever and humorous exploration of love, identity, and deception.

• **As You Like It**: A romantic and lighthearted play that takes place in the Forest of Arden, where the characters escape the complexities of court life. It's a joyful and uplifting tale of love, friendship, and self-discovery.

These three comedies showcase Shakespeare's mastery of language, character development, and storytelling, and continue to be beloved by audiences around the world. 

**<font color='magenta'>Total time:</font>** 13.38 sec.

## Most important battles from WW2

In [ ]:
response = query_model(
    system_message,
    user_message.format(
    adjective="most important",
    number="5",
    items="battles",
    place="WW2"
    ),
    max_length=512)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
What are the most important 5 battles from WW2?
 

**<font color='green'>Answer:</font>** Here are the 5 most important battles of World War II, in no particular order:

• **Battle of Stalingrad (August 1942 - February 1943)**: A decisive turning point on the Eastern Front, where Soviet forces halted the German advance and began pushing them back. The battle marked a shift in momentum in favor of the Allies.

• **Battle of Midway (June 1942)**: A naval battle in the Pacific where the United States defeated Japan, preventing a potential invasion of Hawaii and turning the tide of the war in the Pacific.

• **Battle of El Alamein (October - November 1942)**: A crucial battle in North Africa where British forces defeated the German-Italian army, marking a significant shift in the balance of power in the region and paving the way for the Allied invasion of Italy.

• **Battle of Kursk (July - August 1943)**: The largest tank battle in history, fought on the Eastern Front between German and Soviet forces. The Soviet victory prevented a potential German breakthrough and set the stage for the eventual capture of Berlin.

• **Battle of Normandy (D-Day) (June 6, 1944)**: The Allied invasion of Nazi-occupied France, which marked a major turning point in the war. The successful landing of troops on five beaches in Normandy set the stage for the liberation of Western Europe.

These battles were significant because they:

* Changed the momentum of the war
* Prevented potential German breakthroughs
* Led to the eventual defeat of the Axis powers
* Set the stage for the Allied invasion of Europe 

**<font color='magenta'>Total time:</font>** 22.81 sec.

# Multiple steps reasoning (task chain)

In [ ]:
system_message = """
You are an AI assistant designed to answer questions in a chain of thought.
Use the answer to the first question as input for the second question.
"""
user_message= """
Question one: What are the best city from {country} for tourists?
Question two: What are the best {number} of attractions from the city identified as answer to question one?
Answer:
"""
response = query_model(
    system_message,
    user_message.format(
    number="3",
    country="France"
    ),
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 
Question one: What are the best city from France for tourists?
Question two: What are the best 3 of attractions from the city identified as answer to question one?


**<font color='green'>Answer:</font>**
 

**<font color='green'>Answer:</font>** A great question!

Answer to question one: The best city in France for tourists is Paris, the City of Light.

Answer to question two: As the answer to question one is Paris, the best 3 attractions in Paris are:

1. The Eiffel Tower: An iconic iron lattice tower built for the 1889 World's Fair, offering stunning views of the city from its top level.
2. The Louvre Museum: One of the world's largest and most famous museums, housing an impressive collection of art and artifacts from ancient civilizations to the 19th century, including the Mona Lisa.
3. Notre-Dame Cathedral: A beautiful and historic Gothic cathedral that dates back to the 12th century, famous for its stunning architecture, intricate carvings, and the Hunchback of Notre-Dame novel by Victor Hugo. 

**<font color='magenta'>Total time:</font>** 11.9 sec.

# Reasoning like Einstein will do

In [ ]:
system_message = """
You are a math professor, smart but cool.
Background: A train traveling from Bucharest to Ploiesti (60 km distance) has the speed of 60 km/h.
The train starts in Bucharest and travels until Ploiesti, once, only in this direction.
A swallow, flying with 90 km/h, fly from Ploiesti to the moving train.
When it reaches the train, the swallow flies back toward Ploiesti,
ahead of the train. At Ploiesti turns again back and continues to fly back and forth
(between the train approaching Ploiesti and Ploiesti) until the train reaches Ploiesti.
The swallow will fly continously all the time the train is traveling from Bucharest to Ploiesti.
Reasoning: Think step by step. Explain your reasoning.
Question: {question}
Answer:
"""
t = time()
response = query_model(
    system_message,
    user_message="How many kilometers will travel totally the swallow?",
    temperature=0.1,
    max_length=512)
display(Markdown(colorize_text(f"{response}\n\nTotal time: {round(time()-t, 2)} sec.")))



**<font color='red'>Question:</font>** How many kilometers will travel totally the swallow? 

**<font color='green'>Answer:</font>** Let's break this down step by step.

1. The train travels from Bucharest to Ploiesti, which is a distance of 60 km. Since the train is moving at a constant speed of 60 km/h, it will take 1 hour to cover this distance.
2. The swallow starts flying from Ploiesti to the moving train. Since the train is moving at 60 km/h, the swallow will need to cover a distance of 60 km + the distance it needs to cover to catch up to the train. Let's call this additional distance x. Since the swallow is flying at 90 km/h, it will take it x/90 hours to catch up to the train.
3. Once the swallow catches up to the train, it starts flying back towards Ploiesti, ahead of the train. Since the train is moving at 60 km/h, the swallow will need to cover a distance of 60 km - x to reach Ploiesti. Since the swallow is flying at 90 km/h, it will take it (60 km - x)/90 hours to reach Ploiesti.
4. The swallow will continue to fly back and forth between the train and Ploiesti until the train reaches Ploiesti. Since the train is moving at a constant speed, it will take it 1 hour to cover the remaining distance of x km.
5. During this time, the swallow will make two trips: one from Ploiesti to the train and one from the train to Ploiesti. The total distance it will cover during these two trips is 2x km.
6. Adding up the distances, we get: 60 km (initial distance) + x km (distance to catch up) + (60 km - x) km (distance to reach Ploiesti) + 2x km (distance during the train's remaining journey) = 120 km + x km.

Since the train takes 1 hour to cover the remaining distance of x km, the total time it takes the swallow to fly back and forth is 1 hour. During this time, the swallow will cover a total distance of 120 km + x km.

Now, let's find the value of x. Since the train is moving at 60 km/h, it will cover a distance of 60 km in 1 hour. This means that the distance x is equal to the distance the train covers in the remaining hour, which is 60 km 

**<font color='magenta'>Total time:</font>** 36.01 sec.



**<font color='magenta'>Total time:</font>** 36.01 sec.

In [ ]:
system_message = """
You are an AI assistant designed to answer simple questions.
Please restrict your answer to the exact question asked.
"""

response = query_model(
    system_message,
    user_message="I baked 15 muffins. I ate 2 muffins and gave 5 muffins to a neighbor. My partner then bought 6 more muffins and ate 2. Let's think step-by-step and find out how many muffins do we now have?",
    temperature=1,
    max_length=256)
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** I baked 15 muffins. I ate 2 muffins and gave 5 muffins to a neighbor. My partner then bought 6 more muffins and ate 2. Let's think step-by-step and find out how many muffins do we now have? 

**<font color='green'>Answer:</font>** Let's break it down step by step:

Initially, you baked 15 muffins.

You ate 2 muffins, so you have 15 - 2 = 13 muffins left.

You gave 5 muffins to your neighbor, so you have 13 - 5 = 8 muffins left.

Your partner bought 6 more muffins, making a total of 8 + 6 = 14 muffins.

Your partner ate 2 muffins, so you have 14 - 2 = 12 muffins left.

Therefore, you now have 12 muffins. 

**<font color='magenta'>Total time:</font>** 8.59 sec.

# Conclusions


We compared the results with those with Gemma. Here is the notebook for Gemma: [Prompt Gemma like a Pro](https://www.kaggle.com/code/gpreda/prompt-gemma-like-a-pro).


* Llama3 answered quite decent to most of the questions on history and geography. It looks like an improvement from previous runs of this Notebook.
* Llama3 is good at code generation and math problems.
* The final math problem - failed worst than Gemma (who rationed it's 120 Km; the true answer is 90 km). But actually the rationing is really, really wrong.

In terms of execution time, after I applied the change suggested in one of the comments, it improved very much.

So, comparison of Llama3 8B with Gemma 2B is in favor of Llama3 for general knowledge, maths and code generation, with Gemma slightly better on the more complex math task.

Stay tuned, we will continue with more tests in the next days.